# SEM Particle Analyzer — Colab Quickstart

Upload an SEM image, set the pixel-size calibration, and get a CSV of per-particle measurements plus an annotated overlay.

Repository: [Hj1308/sem-particle-analyzer](https://github.com/Hj1308/sem-particle-analyzer)

## 1. Install dependencies and clone the repo

In [ ]:
!pip install -q tifffile
!git clone https://github.com/Hj1308/sem-particle-analyzer.git
%cd sem-particle-analyzer

## 2. Upload an SEM image

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded))
print(f'Loaded: {image_path}')

## 3. Run the analyzer

Set `pixel_size_nm` from your SEM's scale bar. For example, if the scale bar reads `100 nm` and spans 50 pixels, that's `2.0` nm/pixel.

In [ ]:
from particle_analyzer import analyze

df = analyze(
    image_path,
    output_dir='results',
    pixel_size_nm=2.0,        # ← set this from your SEM
    min_area_px=50,           # raise to ignore tiny noise blobs
    fg_distance_ratio=0.4,    # lower (e.g. 0.3) if touching particles stay merged
)
df.head()

## 4. Inspect results

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

stem = Path(image_path).stem
annotated = cv2.cvtColor(cv2.imread(f'results/{stem}_annotated.tif'), cv2.COLOR_BGR2RGB)
histogram = cv2.cvtColor(cv2.imread(f'results/{stem}_size_distribution.png'), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(annotated); axes[0].set_title('Annotated'); axes[0].axis('off')
axes[1].imshow(histogram); axes[1].set_title('Size distribution'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 5. Summary statistics

In [ ]:
print(f'Particles counted: {len(df)}')
print(df[['eq_diameter_nm', 'circularity', 'aspect_ratio']].describe().round(2))

## 6. Download the results

In [ ]:
from google.colab import files
files.download(f'results/{stem}_measurements.csv')
files.download(f'results/{stem}_annotated.tif')
files.download(f'results/{stem}_size_distribution.png')